# 🏥 Fall Detection - Swift-YOLO for Grove Vision AI V2

**Complete training pipeline**: Dataset → Train → Export → Deploy

**Hardware Target**: Grove Vision AI V2 (WiseEye2 HX6538, Ethos-U55)

**Model**: Swift-YOLO Tiny (192×192, INT8 quantized)

**Classes**: `fall`, `standing`, `sitting`

**Goal**: >90% detection accuracy

---
**Author**: Naron (Minh Tan Tran) | Fall Detection Startup Project

## 1️⃣ Environment Setup

In [ ]:
# Check GPU
!nvidia-smi
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Install dependencies
!pip install -q ultralytics==8.3.0
!pip install -q roboflow
!pip install -q ethos-u-vela
!pip install -q onnx onnxruntime
!pip install -q tensorflow==2.15.0
!pip install -q opencv-python-headless
print('✅ All dependencies installed')

## 2️⃣ Dataset Preparation

Using Roboflow Fall Detection dataset (~10K images)

**Option A**: Use Roboflow API (recommended)

**Option B**: Use SSCMA's built-in datasets

In [ ]:
# ═══════════════════════════════════════════
# OPTION A: Download from Roboflow
# ═══════════════════════════════════════════
from roboflow import Roboflow

# 🔑 Get your API key from https://app.roboflow.com/settings/api
ROBOFLOW_API_KEY = "YOUR_API_KEY"  # ← Replace this

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("roboflow-universe-projects").project("fall-detection-ca3o8")
dataset = project.version(4).download("yolov5", location="./datasets/fall_detection")

print(f'✅ Dataset downloaded: {dataset.location}')

In [ ]:
# Verify dataset structure
import os
import yaml

DATA_DIR = "./datasets/fall_detection"

# Update data.yaml paths
data_yaml_path = os.path.join(DATA_DIR, "data.yaml")
with open(data_yaml_path, 'r') as f:
    data_yaml = yaml.safe_load(f)

print("Dataset config:")
print(f"  Classes: {data_yaml.get('names', [])}")
print(f"  NC: {data_yaml.get('nc', 0)}")

for split in ['train', 'valid', 'test']:
    img_dir = os.path.join(DATA_DIR, split, 'images')
    if os.path.exists(img_dir):
        count = len([f for f in os.listdir(img_dir) if f.endswith(('.jpg','.png'))])
        print(f"  {split}: {count} images")
    else:
        print(f"  {split}: directory not found")

In [ ]:
# ═══════════════════════════════════════════
# Class mapping adjustment
# Map dataset classes to our 3 target classes
# ═══════════════════════════════════════════
import glob

# Check what classes exist in the dataset
class_counts = {}
label_files = glob.glob(os.path.join(DATA_DIR, 'train', 'labels', '*.txt'))

for lf in label_files[:1000]:  # Sample first 1000
    with open(lf, 'r') as f:
        for line in f:
            cls_id = int(line.strip().split()[0])
            class_counts[cls_id] = class_counts.get(cls_id, 0) + 1

print("Class distribution (sampled):")
for cls_id, count in sorted(class_counts.items()):
    name = data_yaml['names'][cls_id] if cls_id < len(data_yaml['names']) else 'unknown'
    print(f"  Class {cls_id} ({name}): {count} annotations")

## 3️⃣ Model Training

Training YOLOv8n (Nano) with parameters optimized for:
- 192×192 input (Grove Vision AI V2 constraint)
- 3 classes: fall, standing, sitting
- Heavy augmentation for small dataset generalization
- Anchor-free detection (better for varied body poses)

In [ ]:
from ultralytics import YOLO

# ═══════════════════════════════════════════
# TRAINING CONFIGURATION
# Optimized for Grove Vision AI V2 deployment
# ═══════════════════════════════════════════

INPUT_SIZE = 192  # MUST be 192 for Grove Vision AI V2 (max 240)
EPOCHS = 300
BATCH_SIZE = 16

# Load YOLOv8 Nano (smallest, best for edge)
model = YOLO('yolov8n.pt')

# Train with edge-optimized settings
results = model.train(
    data=data_yaml_path,
    epochs=EPOCHS,
    imgsz=INPUT_SIZE,
    batch=BATCH_SIZE,
    
    # Learning rate
    lr0=0.01,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=5,
    warmup_momentum=0.8,
    warmup_bias_lr=0.1,
    
    # ── Augmentation (aggressive) ──
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=15.0,         # Falls happen at various angles
    translate=0.1,
    scale=0.5,
    shear=2.0,
    perspective=0.0001,   # Camera perspective variation
    flipud=0.1,           # Inverted camera simulation
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.15,
    copy_paste=0.1,       # Copy-paste augmentation
    
    # ── Training control ──
    patience=50,          # Early stopping patience
    save=True,
    save_period=50,
    plots=True,
    device=0,
    amp=True,             # Mixed precision
    
    # ── Output ──
    project='./runs',
    name='fall_detection_v1',
    exist_ok=True,
)

print("\n✅ Training completed!")

In [ ]:
# View training results
from IPython.display import Image, display
import glob

run_dir = './runs/fall_detection_v1'

# Display training curves
results_img = os.path.join(run_dir, 'results.png')
if os.path.exists(results_img):
    display(Image(filename=results_img, width=900))

# Display confusion matrix
cm_img = os.path.join(run_dir, 'confusion_matrix.png')
if os.path.exists(cm_img):
    display(Image(filename=cm_img, width=600))

# Display PR curve
pr_img = os.path.join(run_dir, 'PR_curve.png')
if os.path.exists(pr_img):
    display(Image(filename=pr_img, width=600))

## 4️⃣ Model Validation & Metrics

In [ ]:
# Validate on test set
best_weights = os.path.join(run_dir, 'weights', 'best.pt')
model = YOLO(best_weights)

# Run validation
val_results = model.val(
    data=data_yaml_path,
    imgsz=INPUT_SIZE,
    batch=BATCH_SIZE,
    split='test',
    conf=0.25,
    iou=0.55,
    plots=True,
)

print("\n" + "="*50)
print("VALIDATION RESULTS")
print("="*50)
print(f"mAP@50:      {val_results.box.map50:.4f} ({val_results.box.map50*100:.1f}%)")
print(f"mAP@50-95:   {val_results.box.map:.4f} ({val_results.box.map*100:.1f}%)")
print(f"Precision:   {val_results.box.mp:.4f} ({val_results.box.mp*100:.1f}%)")
print(f"Recall:      {val_results.box.mr:.4f} ({val_results.box.mr*100:.1f}%)")
print("="*50)

# Per-class results
if hasattr(val_results.box, 'ap50'):
    class_names = data_yaml.get('names', [])
    if isinstance(class_names, dict):
        class_names = [class_names[k] for k in sorted(class_names.keys())]
    print("\nPer-class AP@50:")
    for i, ap in enumerate(val_results.box.ap50):
        name = class_names[i] if i < len(class_names) else f'class_{i}'
        status = '✅' if ap > 0.90 else '⚠️' if ap > 0.80 else '❌'
        print(f"  {status} {name}: {ap:.4f} ({ap*100:.1f}%)")

## 5️⃣ Export to TFLite INT8 (for Grove Vision AI V2)

⚠️ **CRITICAL**: Grove Vision AI V2 **ONLY** accepts `xxx_int8_vela.tflite` format.

Pipeline: `.pt` → ONNX → TFLite (INT8) → Vela (Ethos-U55)

In [ ]:
# ═══════════════════════════════════════════
# Step 5a: Export to TFLite INT8
# ═══════════════════════════════════════════
model = YOLO(best_weights)

# Export with INT8 quantization
tflite_path = model.export(
    format='tflite',
    imgsz=INPUT_SIZE,
    int8=True,
    data=data_yaml_path,  # Used for calibration
)

print(f"\n✅ TFLite INT8 model exported to: {tflite_path}")
print(f"   File size: {os.path.getsize(tflite_path) / 1024:.1f} KB")

In [ ]:
# ═══════════════════════════════════════════
# Step 5b: Compile with Vela for Ethos-U55
# ═══════════════════════════════════════════

# Create Vela config
vela_config = """[System_Config.Ethos_U55_High_End_Embedded]
core_clock=400e6
axi0_port=Sram
axi1_port=Sram

[Memory_Mode.Shared_Sram]
const_mem_area=Axi0
arena_mem_area=Axi1
cache_mem_area=Axi0
arena_cache_size=2097152
"""

with open('vela_config.ini', 'w') as f:
    f.write(vela_config)

# Run Vela compiler
import subprocess

vela_output_dir = './exported_models/vela'
os.makedirs(vela_output_dir, exist_ok=True)

vela_cmd = [
    'vela',
    '--output-dir', vela_output_dir,
    '--accelerator-config', 'ethos-u55-128',
    '--config', 'vela_config.ini',
    '--system-config', 'Ethos_U55_High_End_Embedded',
    '--memory-mode', 'Shared_Sram',
    '--optimise', 'Performance',
    tflite_path,
]

print(f"Running: {' '.join(vela_cmd)}")
result = subprocess.run(vela_cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(f"⚠️ Vela error: {result.stderr}")
else:
    # Find the vela output
    vela_files = glob.glob(os.path.join(vela_output_dir, '*_vela.tflite'))
    if vela_files:
        vela_model = vela_files[0]
        print(f"\n✅ Vela model ready: {vela_model}")
        print(f"   File size: {os.path.getsize(vela_model) / 1024:.1f} KB")
        print(f"\n📦 This file is ready for deployment to Grove Vision AI V2!")

## 5c: Alternative - Use SSCMA Export Pipeline

If you prefer SSCMA's official export (recommended for guaranteed compatibility):

In [ ]:
# ═══════════════════════════════════════════
# Alternative: SSCMA official pipeline
# ═══════════════════════════════════════════

# Clone SSCMA
!git clone https://github.com/Seeed-Studio/ModelAssistant.git
%cd ModelAssistant
!pip install -r requirements.txt

# Export using SSCMA tools
# This ensures 100% compatibility with Grove Vision AI V2
!python3 tools/export.py \
    configs/swift_yolo/swift_yolo_tiny_1xb16_300e_coco.py \
    {best_weights} \
    --format tflite \
    --cfg-options \
        imgsz="({INPUT_SIZE},{INPUT_SIZE})" \
        num_classes=3

## 6️⃣ Test on Video

In [ ]:
# ═══════════════════════════════════════════
# Video evaluation with temporal smoothing
# ═══════════════════════════════════════════
import cv2
import numpy as np
from collections import deque

def evaluate_video(model_path, video_path, conf=0.25, buffer_size=5):
    """Evaluate fall detection on video with temporal smoothing."""
    model = YOLO(model_path)
    cap = cv2.VideoCapture(video_path)
    
    fps = cap.get(cv2.CAP_PROP_FPS)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    detection_buffer = deque(maxlen=buffer_size)
    results_log = []
    fall_frames = 0
    total_frames = 0
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        # Inference
        preds = model.predict(frame, imgsz=INPUT_SIZE, conf=conf, verbose=False)
        
        # Check for fall in this frame
        has_fall = False
        for r in preds:
            for box in r.boxes:
                cls_name = r.names[int(box.cls[0])]
                if 'fall' in cls_name.lower():
                    has_fall = True
                    break
        
        detection_buffer.append(has_fall)
        
        # Temporal voting (60% threshold)
        if sum(detection_buffer) >= len(detection_buffer) * 0.6:
            fall_frames += 1
            results_log.append(('FALL', total_frames / fps))
        
        total_frames += 1
    
    cap.release()
    
    print(f"\nResults for: {video_path}")
    print(f"  Total frames: {total_frames}")
    print(f"  Fall frames (smoothed): {fall_frames}")
    print(f"  Fall percentage: {fall_frames/total_frames*100:.1f}%")
    
    return results_log

# Upload a test video or use sample
# from google.colab import files
# uploaded = files.upload()
# video_path = list(uploaded.keys())[0]

# evaluate_video(best_weights, video_path)

## 7️⃣ Deploy to Grove Vision AI V2

### Method 1: SenseCraft Web Toolkit (No-Code)
1. Go to https://seeed-studio.github.io/SenseCraft-Web-Toolkit/
2. Connect your Grove Vision AI V2 via USB-C
3. Click **Upload Custom Model**
4. Select the `*_int8_vela.tflite` file
5. Press **Send**

### Method 2: Command Line (xmodem)
```bash
python3 xmodem/xmodem_send.py \
    --port=/dev/ttyACM0 \
    --baudrate=921600 \
    --protocol=xmodem \
    --file=we2_image_gen_local/output_case1_sec_wlcsp/output.img \
    --model="fall_detection_int8_vela.tflite 0xB7B000 0x00000"
```

### Method 3: Edge Impulse
Upload via Edge Impulse Studio → Deploy → Grove Vision AI V2

In [ ]:
# Download all artifacts
from google.colab import files
import shutil

# Package everything
export_dir = './fall_detection_package'
os.makedirs(export_dir, exist_ok=True)

# Copy model files
shutil.copy2(best_weights, os.path.join(export_dir, 'best.pt'))
if os.path.exists(tflite_path):
    shutil.copy2(tflite_path, export_dir)

vela_files = glob.glob('./exported_models/vela/*_vela.tflite')
for vf in vela_files:
    shutil.copy2(vf, export_dir)

# Copy training results
for img in ['results.png', 'confusion_matrix.png', 'PR_curve.png']:
    src = os.path.join(run_dir, img)
    if os.path.exists(src):
        shutil.copy2(src, export_dir)

# Zip and download
shutil.make_archive(export_dir, 'zip', export_dir)
files.download(f'{export_dir}.zip')
print('\n📦 Package downloaded! Ready for deployment.')